# 🛰️ SVAMITVA Feature Extraction — Production GPU Training

**Features:** Buildings (roof type), Roads, Waterbodies, Utilities

**Setup:** Upload your `DATASET` folder to Google Drive root, then run all cells.

## 1. GPU Check

In [1]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)')
else:
    print('⚠️ No GPU! Runtime > Change runtime type > GPU')
    raise RuntimeError('GPU required')

PyTorch: 2.9.0+cu128
CUDA: True
GPU: Tesla T4 (15.6 GB)


## 2. Install Dependencies

In [2]:
%%capture
!pip install -q rasterio geopandas shapely pyproj fiona
!pip install -q 'albumentations>=2.0' opencv-python scikit-image
!pip install -q tensorboard tqdm timm
!pip install -q segmentation-models-pytorch
!pip install -q 'numpy>=1.24,<2.0'
print('✅ Done')

## 3. Mount Drive & Clone Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/DATASET'

## 3b. Clone Project

In [ ]:
import os, sys

PROJECT = '/content/svamitva_model'
!rm -rf {PROJECT}
!git clone https://github.com/aaronrduk/geoii.git {PROJECT}

%cd {PROJECT}
sys.path.insert(0, PROJECT)

# Verify key files exist
!ls prepare_dataset.py train.py training/config.py

# Check dataset on Drive
DATASET_PATH = '/content/drive/MyDrive/DATASET'
if os.path.exists(DATASET_PATH):
    print(f'\n\u2705 Dataset found: {DATASET_PATH}')
    for item in sorted(os.listdir(DATASET_PATH)):
        full = os.path.join(DATASET_PATH, item)
        if os.path.isdir(full):
            print(f'   \U0001f4c1 {item}/ ({len(os.listdir(full))} files)')
else:
    print('\u26a0\ufe0f DATASET not found on Drive. Mount Drive first or upload DATASET.')

## 4. Prepare Dataset

In [ ]:
import os

!python prepare_dataset.py --dataset_dir "{DATASET_PATH}" --output_dir dataset/train_full --copy

train_dir = 'dataset/train_full'
if os.path.exists(train_dir):
    villages = [d for d in sorted(os.listdir(train_dir))
                if os.path.isdir(os.path.join(train_dir, d))]
    print(f'\n✅ {len(villages)} villages ready')
    for v in villages:
        vp = os.path.join(train_dir, v)
        ortho = os.path.exists(os.path.join(vp, 'orthophoto.tif'))
        anno = os.path.join(vp, 'annotations')
        n = len([f for f in os.listdir(anno) if f.endswith('.shp')]) if os.path.exists(anno) else 0
        print(f'   {v}: ortho={ortho}, annotations={n}')

## 5. Training Config

In [ ]:
from training.config import TrainingConfig

config = TrainingConfig(
    train_data_dir='dataset/train_full',
    val_data_dir=None,
    checkpoint_dir='checkpoints',
    log_dir='logs',
    backbone='resnet50',
    pretrained=True,
    batch_size=4,
    num_epochs=50,
    learning_rate=1e-4,
    weight_decay=1e-4,
    scheduler='cosine',
    mixed_precision=True,
    num_workers=2,
    image_size=512,
    early_stopping=True,
    patience=15,
)

print('✅ Config:')
for k, v in vars(config).items():
    print(f'   {k}: {v}')

## 6. Train 🚀

In [ ]:
from train import main as train_main
train_main(config)

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

## 8. Evaluate

In [ ]:
import torch, matplotlib.pyplot as plt
from models.feature_extractor import FeatureExtractorModel
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FeatureExtractorModel(backbone='resnet50', pretrained=False)

ckpt = Path('checkpoints/best_model.pth')
if ckpt.exists():
    state = torch.load(ckpt, map_location=device, weights_only=False)
    model.load_state_dict(state.get('model_state_dict', state), strict=False)
    print(f'✅ Loaded (epoch {state.get("epoch", "?")})')
model.to(device).eval()

dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    out = model(dummy)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (k, t) in enumerate(zip(
    ['building_mask', 'road_mask', 'waterbody_mask', 'utility_mask'],
    ['Buildings', 'Roads', 'Water', 'Utilities']
)):
    axes[i].imshow(out[k].cpu().numpy()[0, 0], cmap='hot')
    axes[i].set_title(t); axes[i].axis('off')
plt.tight_layout(); plt.show()

## 9. Save Checkpoints to Drive

In [ ]:
import shutil
from pathlib import Path

save_dir = Path('/content/drive/MyDrive/svamitva_trained')
save_dir.mkdir(parents=True, exist_ok=True)

for folder in ['checkpoints', 'logs']:
    src, dst = Path(folder), save_dir / folder
    if src.exists():
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'✅ {folder} → Drive')

print(f'🏁 Saved to {save_dir}')